# UN Comtrade データ取得

分類は **HS (as reported)** = `clCode="HS"`（各国が報告した時点の HS 版をそのまま使う）。

**設定セル（「ここを書き換える」の箇所）だけ書き換えれば取得対象が変わる。** 他のセルは触らなくてよい。

## 1. セットアップ

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

import comtradeapicall as c

load_dotenv(Path.home() / ".comtrade_env")
KEY = (os.environ.get("COMTRADE_KEY") or "").strip()
if not KEY:
    raise RuntimeError("COMTRADE_KEY が未設定。~/.comtrade_env を確認すること。")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print(f"キー検出（{len(KEY)} 文字）")

## 2. コード解決ユーティリティ

国は **数値コード / ISO3 / 国名** のどれで書いてもよい。内部で数値コードに変換する。
`partner` の参照表だけ列名が大文字始まり（`PartnerCode`）なので、そこも吸収している。

In [ ]:
_REF = {}


def _ref(kind):
    """reporter / partner の参照表を取得してキャッシュする（列名の揺れを吸収）。"""
    if kind not in _REF:
        r = c.getReference(kind)
        # partner 表は PartnerCode / PartnerCodeIsoAlpha3 と大文字始まり
        cols = {col.lower(): col for col in r.columns}
        _REF[kind] = (r, cols[f"{kind}code"], cols[f"{kind}codeisoalpha3"], cols["text"])
    return _REF[kind]


def resolve(values, kind):
    """国の指定を数値コードのカンマ区切り文字列に変換する。

    None                -> None（＝全件）
    "392" / 392         -> "392"
    "JPN" / "jpn"       -> "392"
    "Japan"             -> "392"
    ["JPN", "USA"]      -> "392,842"
    """
    if values is None:
        return None
    if isinstance(values, (str, int)):
        values = [values]

    ref, col_code, col_iso, col_text = _ref(kind)
    out = []
    for v in values:
        s = str(v).strip()
        if not s:
            continue
        if s.isdigit():                      # 数値コードはそのまま
            out.append(s)
            continue
        up = s.upper()
        hit = ref[ref[col_iso].astype(str).str.upper() == up]          # ISO3 完全一致
        if hit.empty:
            hit = ref[ref[col_text].astype(str).str.upper() == up]     # 国名 完全一致
        if hit.empty:                                                   # 部分一致で候補提示
            cand = ref[ref[col_text].astype(str).str.contains(s, case=False, na=False)]
            raise ValueError(
                f"{kind} '{s}' を解決できない。候補: "
                + (", ".join(f"{r[col_text]}={r[col_code]}" for _, r in cand.head(8).iterrows())
                   or "なし"))
        out.append(str(hit.iloc[0][col_code]))
    return ",".join(out) if out else None


def as_list(v):
    """"a" / "a,b" / ["a","b"] を ["a","b"] に正規化する。"""
    if v is None:
        return []
    if isinstance(v, (str, int)):
        v = str(v).split(",")
    return [str(x).strip() for x in v if str(x).strip()]

## 3. HS コードを調べる

`clCode="HS"`（Combined HS）で使えるコードは **`getReference("cmd:HS")` で全件取れる**（8,262件）。

| `aggrLevel` | 件数 | 内容 |
|---|---|---|
| 0 | 1 | `TOTAL` 総計 |
| 2 | 98 | 章（例 `01` Animals; live） |
| 4 | 1,266 | 項（例 `0201` Meat of bovine animals; fresh or chilled） |
| 6 | 6,897 | 号（例 `020110` 枝肉・半丸枝肉） |

`isLeaf=1` が末端（それ以上分割されない）コード。`parent` で上位コードを辿れる。

**Combined HS は HS1992〜HS2022 の全版を束ねたもの**なので、版によって統廃合された
コードも混在する（`0101` 配下に旧 `010110` と新 `010111`/`010119` が並ぶ等）。
特定版に固定したい場合は `CL` を `"H6"`(HS2022) や `"H5"`(HS2017) 等に変える。

このほか、表には載らないが **集計レベル指定子**が `CMD` に使える。

| 値 | 意味 |
|---|---|
| `"TOTAL"` | 全品目の総計 |
| `"AG2"` | 全2桁（97コード） |
| `"AG4"` | 全4桁 |
| `"AG6"` | 全6桁（4,840コード・件数が多く上限に当たりやすい） |

In [ ]:
HS = c.getReference("cmd:HS")     # 8,262 行
print(f"{len(HS):,} 件  列: {list(HS.columns)}")


def find_hs(keyword=None, prefix=None, level=None, leaf_only=False):
    """HS コードを検索する。

    keyword   : 品名に含まれる語（大小文字は無視）
    prefix    : コードの先頭一致（例 "0201"）
    level     : 桁数 2 / 4 / 6
    leaf_only : 末端コードのみに絞る
    """
    r = HS
    if keyword:
        r = r[r["text"].astype(str).str.contains(keyword, case=False, na=False)]
    if prefix:
        r = r[r["id"].astype(str).str.startswith(str(prefix))]
    if level is not None:
        r = r[r["aggrLevel"] == level]
    if leaf_only:
        r = r[r["isLeaf"] == 1]
    return r[["id", "text", "parent", "isLeaf", "aggrLevel", "standardUnitAbbr"]]


# 例: 0201 系列（牛肉・生鮮/冷蔵）
find_hs(prefix="0201")

In [ ]:
# 例: 品名から探す
find_hs(keyword="bovine", level=6).head(15)

In [ ]:
# 全件を手元に置いておくと grep や Excel でも探せる
Path("data").mkdir(exist_ok=True)
HS_CSV = Path("data") / "hs_codes.csv"
HS.to_csv(HS_CSV, index=False, encoding="utf-8-sig")
print("保存:", HS_CSV.resolve(), f"({len(HS):,} 行)")

### 保存した CSV の中身を見る

読み戻すときは **`dtype={"id": str, "parent": str}` を必ず指定する**。
指定しないと `0201` が `201` になり、先頭ゼロが消えて品目コードとして使えなくなる。

In [ ]:
hs_saved = pd.read_csv(HS_CSV, dtype={"id": str, "parent": str})

print(f"{len(hs_saved):,} 行 x {hs_saved.shape[1]} 列")
print(hs_saved.dtypes.to_string())
print("\n欠損:", hs_saved.isna().sum().to_dict())   # standardUnitAbbr は n/a 品目で欠損
hs_saved.head(20)

In [ ]:
from IPython.display import HTML, display


def show_table(df, max_rows=200, height=450):
    """スクロールできる表として表示する。max_rows=None で全行。"""
    d = df if max_rows is None else df.head(max_rows)
    note = f"{len(d):,} / {len(df):,} 行を表示"
    display(HTML(
        f'<div style="margin-bottom:4px; font-size:90%">{note}</div>'
        f'<div style="max-height:{height}px; overflow:auto; border:1px solid #ccc">'
        + d.to_html(index=False) + "</div>"))


show_table(hs_saved)            # 先頭200行をスクロール表示
# show_table(hs_saved, None)    # 全8,262行（描画が重いので必要なときだけ）

In [ ]:
# 目的の行だけ抜き出して見るほうが実用的
show_table(hs_saved[hs_saved["aggrLevel"] == 2], max_rows=None)   # 章98件だけ全表示

## 4. 取得条件 — ここだけ書き換える

| 変数 | 意味 | 書き方の例 |
|---|---|---|
| `CMD` | **HS 品目** | `"020110"` / `"0201.10"`（ドット可） / `["020110", "020120"]` / `"AG6"` 全6桁 / `"TOTAL"` 総計 |
| `REPORTER` | **報告国** | `None` 全report国 / `"JPN"` / `"Japan"` / `"392"` / `["JPN", "USA", "AUS"]` |
| `FLOW` | **貿易フロー** | `"M"` 輸入 / `"X"` 輸出 / `["M", "X"]` 両方 |
| `PARTNER` | **相手国** | `None` 全相手国 / `"0"` World のみ / `["USA", "CHN"]` |
| `PERIOD` | **期間** | `"2022"` / `["2020", "2021", "2022"]` / 月次 `"202201"`（`FREQ="M"`） |

`FREQ` は `"A"` 年次 / `"M"` 月次。月次にしたら `PERIOD` も `"202201"` 形式にすること。

In [ ]:
# ===== ここを書き換える =========================================
CMD      = "0201.10"          # HS 品目（ドットは自動で除去）
REPORTER = None               # None = 全報告国
FLOW     = "X"                # 輸出（fetch_ch01-24.ipynb と統一）
PARTNER  = None               # None = 全相手国
PERIOD   = "2022"             # 年次
FREQ     = "A"                # "A" 年次 / "M" 月次
# ===============================================================

# 合計行だけを取る指定（内訳行の混入＝二重計上を防ぐ。None にしないこと）
PARTNER2 = "0"                # World
CUSTOMS  = "C00"              # TOTAL CPC
MOT      = "0"                # TOTAL MOT
CL       = "HS"               # HS (as reported)

# --- 正規化 ---------------------------------------------------
cmd_codes    = ",".join(x.replace(".", "") for x in as_list(CMD))
flow_codes   = ",".join(x.upper() for x in as_list(FLOW))
periods      = as_list(PERIOD)
reporter_cds = resolve(REPORTER, "reporter")
partner_cds  = resolve(PARTNER, "partner")

print(f"品目    : {cmd_codes}")
print(f"報告国  : {reporter_cds or '全報告国'}")
print(f"フロー  : {flow_codes}")
print(f"相手国  : {partner_cds or '全相手国'}")
print(f"期間    : {periods}  (freq={FREQ})")

## 5. 取得

### ⚠️ 押さえてある2つの罠

1. **100,000件の暗黙の打ち切り** — `getFinalData` は `maxRecords` に何を指定しても
   1回10万件で切り、警告も例外も出さない。取得前に `getCountFinalData` で真の件数を
   確認し、超える場合は期間ごとに自動分割する。
2. **内訳行による二重計上** — `partner2Code` / `customsCode` / `motCode` を `None` に
   すると、輸送モード別・通関手続別・原産国別の内訳が合計行と一緒に返る。実測では
   551行が2,613行に膨れた。上の合計コード指定で回避している。

In [ ]:
# サーバ側の1リクエスト上限（実測値）。maxRecords に 250,000 を指定しても
# 100,000 件で打ち切られ、警告も例外も出ないため自前で見張る。
# 実測例: 真の件数 177,693 → 取得 100,000（77,693 件が黙って欠落）
CAP = 100_000


def _args(period_arg):
    return dict(
        typeCode="C", freqCode=FREQ, clCode=CL, period=period_arg,
        reporterCode=reporter_cds, cmdCode=cmd_codes, flowCode=flow_codes,
        partnerCode=partner_cds, partner2Code=PARTNER2,
        customsCode=CUSTOMS, motCode=MOT,
    )


def count_records(period_arg):
    res = c.getCountFinalData(KEY, **_args(period_arg))
    if res is None or getattr(res, "empty", True):
        return None
    return int(res["count"].iloc[0])


def fetch_one(period_arg):
    """1リクエスト分を取得し、打ち切りを検出したら例外を出す。"""
    expected = count_records(period_arg)
    df_p = c.getFinalData(KEY, maxRecords=CAP, includeDesc=True, **_args(period_arg))
    got = 0 if df_p is None or getattr(df_p, "empty", True) else len(df_p)

    label = f"  {period_arg}: {got:,} 行"
    if expected is not None:
        label += f" / 真 {expected:,}"
    print(label)

    # 真の件数が分かるならそれを正とする（ちょうど CAP 件でも欠落がなければ通す）
    if expected is not None:
        if got < expected:
            raise RuntimeError(
                f"★打ち切り: {period_arg} は取得 {got:,} < 真 {expected:,}。"
                " REPORTER や PARTNER を絞るか、CMD や PERIOD を分割すること。")
    elif got >= CAP:
        raise RuntimeError(
            f"★打ち切りの疑い: {period_arg} が上限 {CAP:,} ちょうど、かつ真の件数を"
            " 取得できていない。条件を絞って再実行すること。")
    return df_p if got else None


# 全期間まとめて上限に収まるなら1回、超えるなら期間ごとに分割
joined = ",".join(periods)
total = count_records(joined)
print(f"該当件数（全期間）: {total:,}" if total is not None else "件数取得に失敗")

if total is not None and total < CAP:
    frames = [fetch_one(joined)]
else:
    print(f"上限 {CAP:,} を超えるため期間ごとに分割")
    frames = []
    for i, p in enumerate(periods):
        frames.append(fetch_one(p))
        if i < len(periods) - 1:
            time.sleep(1.0)   # レート制限対策

frames = [f for f in frames if f is not None]
if not frames:
    raise RuntimeError("データが取得できなかった（条件・権限・レート制限を確認）")

df = pd.concat(frames, ignore_index=True)
print(f"\n取得: {len(df):,} 行 x {df.shape[1]} 列")

## 6. 検証 — 集計する前に必ず通す

In [ ]:
UNIQ = ["period", "reporterCode", "partnerCode", "cmdCode", "flowCode"]
dup = df.duplicated(subset=UNIQ, keep=False).sum()

world = df[df["partnerDesc"] == "World"]

print(f"重複行       : {dup}  (0 であること)")
print(f"品目コード   : {sorted(df['cmdCode'].astype(str).unique())[:10]}")
print(f"期間         : {sorted(df['period'].astype(str).unique())}")
print(f"報告国数     : {df['reporterCode'].nunique()}")
print(f"相手国数     : {df['partnerCode'].nunique()}")
print(f"フロー       : {sorted(df['flowDesc'].astype(str).unique())}")

if dup:
    raise RuntimeError(
        "内訳行が混入している。PARTNER2 / CUSTOMS / MOT が合計コードか確認すること。")
print("✓ 重複なし — そのまま集計してよい")

In [ ]:
COLS = ["period", "reporterCode", "reporterDesc", "flowDesc",
        "partnerCode", "partnerDesc", "cmdCode", "cmdDesc",
        "qty", "qtyUnitAbbr", "netWgt", "primaryValue"]

df[[col for col in COLS if col in df.columns]].head(20)

In [ ]:
# 報告国別の合計（partnerDesc == "World" が全世界合計の行）
(world.groupby(["period", "reporterDesc"], as_index=False)["primaryValue"].sum()
      .nlargest(20, "primaryValue")
      .assign(million_USD=lambda d: (d["primaryValue"] / 1e6).round(2))
      .reset_index(drop=True))

## 7. 保存

In [ ]:
OUT = Path("data")
OUT.mkdir(exist_ok=True)

tag = "_".join(filter(None, [
    "hs" + cmd_codes.replace(",", "-"),
    flow_codes.replace(",", "-"),
    (reporter_cds or "allrep").replace(",", "-"),
    "-".join(periods),
]))
path = OUT / f"{tag}.csv"
df.to_csv(path, index=False, encoding="utf-8-sig")   # Excel で開くなら utf-8-sig
print("保存:", path.resolve(), f"({len(df):,} 行)")

## メモ

- `PARTNER2` / `CUSTOMS` / `MOT` を `None` にすると内訳行が混ざり**合計が数倍に膨れる**。
  合計コード（`"0"` / `"C00"` / `"0"`）のままにすること。
- `partnerDesc == "World"` は全世界合計。個別相手国と合算すると**二重計上**になる。
- `primaryValue` は USD（名目）、`netWgt` は kg、`qty` の単位は `qtyUnitAbbr` を見る。
- `period` は文字列型。比較は `df["period"] == "2022"` と書く。
- 国名が解決できないと候補付きの `ValueError` が出る。表記は
  `c.getReference("reporter")` で確認できる。
- 学内プロキシ配下では各関数に `proxy_url=` を渡す。